In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

In [10]:
df = pd.read_csv("../data/processed/cleaned_loans.csv")
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,sub_grade,emp_title,...,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding
0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B2,NaN,...,False,False,False,False,False,False,False,False,False,False
1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C4,Ryder,...,False,False,False,False,False,False,False,False,False,False
2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C5,NaN,...,False,False,False,False,False,False,False,True,False,False
3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C1,AIR RESOURCES BOARD,...,False,False,False,False,False,True,False,False,False,False
4,1075269,1311441,5000,5000,5000.0,36 months,7.90,156.46,A4,Veolia Transportaton,...,False,False,False,False,False,False,False,False,False,True


In [11]:
default_rate = df["is_default"].mean()*100
print(f"Total default rate is {default_rate}")

Total default rate is 14.586411592399617


# Observation
Approximately 14.6% of loans resulted in default, indicating moderate credit risk exposure within the portfolio.

In [47]:
leakage_cols = [
    'recoveries',
    'collection_recovery_fee',
    'total_rec_late_fee',
    'total_pymnt',
    'total_pymnt_inv',
    'total_rec_prncp',
    'last_pymnt_amnt',
    "verification_status_Not Verified"
]
df_no_leak = df.drop(columns=[col for col in leakage_cols if col in df.columns])
corr = df_no_leak.corr(numeric_only=True)["is_default"]
corr = corr.drop("is_default").dropna()
positive_corr = corr[corr > 0].sort_values(ascending=False)
negative_corr = corr[corr < 0].sort_values()

print("Top Positive Correlations (Higher Default Risk):")
print(positive_corr.head(5))

print("\nTop Negative Correlations (Lower Default Risk):")
print(negative_corr.head(5))

Top Positive Correlations (Higher Default Risk):
int_rate       0.211390
term_months    0.173487
revol_util     0.099870
grade_E        0.094605
grade_F        0.082607
Name: is_default, dtype: float64

Top Negative Correlations (Lower Default Risk):
grade_A                  -0.144456
grade_B                  -0.044435
purpose_credit_card      -0.041724
annual_inc               -0.040867
purpose_major_purchase   -0.029327
Name: is_default, dtype: float64


# Observation
1. Higher interest rate leads to higher default rate as it is strongly correlated
2. Also when we increase the term month the chances of default increases
3. Grade A and Grade B  are most negative correlated so whenever someone belongs to that category the chances of default are low
4. Higher the annual income lower the risk of default 

In [19]:
df.groupby("term_months")["is_default"].mean() * 100

term_months
36.0    11.090872
60.0    25.313785
Name: is_default, dtype: float64

# Observation
Higher tenure lead to more default rate and low tenure reduces defualt rate

In [20]:
grade_cols = [col for col in df.columns if col.startswith("grade_")]
for col in grade_cols:
    rate = df[df[col] == 1]["is_default"].mean() * 100
    print(f"{col}: {rate:.2f}%")

grade_A: 5.99%
grade_B: 12.21%
grade_C: 17.19%
grade_D: 21.99%
grade_E: 26.85%
grade_F: 32.68%
grade_G: 33.78%


# Observation
Grade A has lowest defualt rate and Grade G has highest rate of default 